In [ ]:
import pandas as pd

import re 

import matplotlib.pyplot as plt
import math
import glob
import matplotlib.ticker


In [ ]:
plt.rcParams["figure.figsize"] = (15,10)
plt.rcParams['figure.dpi'] = 150

In [ ]:
exprparts  = re.compile(r'Particles generated: max: (\d+) points .* sum : (\d+) .*mean : (\d+) ')
exprts30k  = re.compile(r'.*        timestep\W+(\d+(.\d+)?)s.*+',  flags=re.DOTALL)
exprsync30k  = re.compile(r'.*      sync\W+(\d+(.\d+)?)s.*+',  flags=re.DOTALL)
exprgrid30k  = re.compile(r'.*      grid service\W+(\d+(.\d+)?)s.*+',  flags=re.DOTALL)
exprmaxRun  = re.compile(r'Runtime max: (\d+(.\d+)?)')
# Particles generated: max: 51363 points @ rank: 88 | min: 38133 points @ rank: 110 | sum : 11439567 | mean : 44685 | median : 44396

def slurmToDF(name):
    f = open(name, "r")
    str =f.read()

    splt = str.split("Mesh load time at rank ")

    times = []
    rankId = 0

    for s in splt:

        found = True

        if len(s) < 1:
            continue


        split2 = s.split("timestep done 30000")
        # print(split2)
        if len(split2) <= 1:
            continue

        # print(s, split2)

        line = {}
        parts = exprparts.findall(split2[0])
        ts30k = exprts30k.findall(split2[1])
        sync30k = exprsync30k.findall(split2[1])
        grid30k = exprgrid30k.findall(split2[1])
        maxRun = exprmaxRun.findall(split2[1])
        if(len(maxRun) == 0):
            continue

        if(len(maxRun) >= 1 and len(ts30k) >= 1):
            try :
                line["runtime"] =  maxRun[0][0]
                line["timestep30k"] =  ts30k[0][0]
                line["sync30k"] =  sync30k[0][0]
                line["grid30k"] =  grid30k[0][0]
                line["totParticles"] =  parts[0][1]
                line["perRankParticles"] =  parts[0][2]
                line["ranks"] = math.floor(pd.to_numeric(parts[0][1])/pd.to_numeric(parts[0][2])) ##  not accurate
                times.append(line)

                rankId = (rankId+1)%4
            except:
                pass
        pass

    df = pd.DataFrame.from_dict(times)
    df["runtime"] = pd.to_numeric(df["runtime"])
    df["timestep30k"] = pd.to_numeric(df["timestep30k"])
    df["sync30k"] = pd.to_numeric(df["sync30k"])
    df["grid30k"] = pd.to_numeric(df["grid30k"])
    df["totParticles"] = pd.to_numeric(df["totParticles"])
    df["perRankParticles"] = pd.to_numeric(df["perRankParticles"])
    return df

#dfPost = slurmToDF("slurm-8076586.out")

dfPre = slurmToDF("Lumi/output/out_160k-10283003.out")

In [ ]:
allPaths = glob.glob("Lumi/output/out_*")

latest = {}

pathSplit = re.compile(r'out_(\d+)k-(\d+).out')
for f in allPaths:
    split = pathSplit.findall(f)
    print(split)
    case = split[0][0]
    jobid = split[0][1]
    if (case in latest) :
        if (latest[case] < jobid):
              latest[case] = jobid
    else :
        latest[case] = jobid
     
    pass



In [ ]:
fig, ax = plt.subplots()

frames = []
for p in latest:
    print(p)
    dfPre = slurmToDF("Lumi/output/out_"+p+"k-"+latest[p]+".out")
    frames.append(dfPre)

df = pd.concat(frames)
groups = df.groupby(["totParticles"])

for name, group in groups:
    print(group)
    #ax.plot(group["ranks"], group["timestep30k"], marker='o', linestyle='-', ms=10, label=name)
    #ax.plot(group["ranks"], group["runtime"], marker='o', linestyle='-', ms=10, label=name)
    ax.plot(group["ranks"], group["runtime"], marker='o', linestyle='-', ms=10, label=name)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log", base=10)
    

ax.set_xticks([1,2,4,8,16,32,64,128,256,512,1024,2048,4096])
ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
plt.grid()
ax.grid(which='minor', color='#EEEEEE', linestyle='-', linewidth=0.5)
ax.get_yaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())

ax.set_xlabel("Num ranks")
ax.set_ylabel("Runtime (s)")


ax.legend()


In [ ]:
fig, ax = plt.subplots()

frames = []
for p in latest:
    print(p)
    dfPre = slurmToDF("Lumi/output/out_"+p+"k-"+latest[p]+".out")
    frames.append(dfPre)

df = pd.concat(frames)
groups = df.groupby(["totParticles"])

for name, group in groups:
    print(group)
    ax.plot(group["ranks"], group["timestep30k"], marker='o', linestyle='-', ms=10, label=name)
    #ax.plot(group["ranks"], group["runtime"], marker='o', linestyle='-', ms=10, label=name)
    #ax.plot(group["ranks"], group["runtime"], marker='o', linestyle='-', ms=10, label=name)
    ax.set_xscale("log", base=2)
    ax.set_yscale("log", base=10)
    

ax.set_xticks([1,2,4,8,16,32,64,128,256,512,1024,2048,4096])
ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
plt.grid()
ax.grid(which='minor', color='#EEEEEE', linestyle='-', linewidth=0.5)
ax.get_yaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())

ax.set_xlabel("Number of ranks (and GPUs)")
ax.set_ylabel("Runtime for 5000 iterations (s)")


ax.legend()

In [ ]:
def compare(jobs, basePath):
    fig, ax = plt.subplots()
    frames = []
    for p in jobs:
        print(p)
        dfPre = slurmToDF(basePath+p)

        # ax.plot(dfPre["perRankParticles"], dfPre["timestep30k"], marker='o', linestyle='-', ms=10, label=name)
        ax.plot(dfPre["ranks"], dfPre["runtime"], marker='o', linestyle='-', ms=10, label=p)
        # ax.plot(dfPre["ranks"], dfPre["timestep30k"], marker='o', linestyle='-', ms=10, label=name)
        ax.set_xscale("log", base=2)
        ax.set_yscale("log", base=10)
    
    ax.set_xlabel("Num ranks")
    ax.set_ylabel("Runtime (s)")

    ax.legend()

In [ ]:

jobs = ["out_2800k-10298200.out","out_2800k-10283305.out", 
        "out_2800k-10298734.out", "out_2800k-10298831.out", 
        "out_2800k-10299583.out", "out_2800k-10318978.out", 
        "out_2800k-10320700.out"]

jobs = ["out_2800k-10432660.out", "out_2800k-10435745.out",
        "out_2800k-10435764.out", "out_2800k-10435790.out",
        "out_2800k-10435872.out", "out_2800k-10436450.out",
        "out_2800k-10436481.out", "out_2800k-10436846.out",
        "out_2800k-10436913.out", "out_2800k-10436918.out",
        "out_2800k-10439651.out", "out_2800k-10439766.out",
        "out_2800k-10443264.out"
        ]

compare(jobs, "Lumi/output/")

In [ ]:

jobs = ["out_38600k-10320328.out", "out_38600k-10322063.out", "out_38600k-10444161.out",
        "out_38600k-10445037.out", "out_38600k-10445075.out", "out_38600k-10445388.out"]

compare(jobs, "Lumi/output/")

In [ ]:


allPaths = glob.glob("Lumi/output/out_310k*")

compare(allPaths, "")